# Activation Patching

Bare-bones activation patching from a LoRA-adapted model into the same base model with the adapter disabled. One forward pass with LoRA on caches an activation at a chosen `(layer, component, token position)`; a second pass with LoRA off generates text with that activation patched in.

`disable_adapter()` lets us treat donor (LoRA on) and recipient (LoRA off) as the same module tree, so a single pytorch forward hook does both jobs.

Selectable components: `mlp`, `attn`, `resid`, `gate_proj`, `up_proj`, `down_proj`.

In [1]:
import sys
from contextlib import nullcontext
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import torch
from loguru import logger

from sl.utils.model_selection import load_registry, resolve_model_selection

bundle = load_registry()
ARTIFACTS_DIR = bundle.artifacts_dir
reg = bundle.registry

logger.info(f"Registry: {bundle.registry_path}  experiments={len(reg['experiments'])}")

2026-04-30 11:15:28.769 | INFO     | __main__:<module>:17 - Registry: /net/projects2/interp/subliminal/shared/results/registry.json  experiments=7520


## Helpers

All function definitions used by the rest of the notebook: prompt rendering / token tables and the activation-patching primitives (`cache_activations`, `make_patch_hook`, `patched_generate`, `top_k_next`).

These functions reference `model`, `tokenizer`, and `decoder_layers` only at *call time*, so it's safe to define them up here before the model is loaded — just don't call them until after section 2.

In [2]:
# --- Prompt rendering / tokenization ---

def render(user: str, system: str | None = None) -> str:
    msgs = []
    if system is not None:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": user})
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def token_table(user: str, system: str | None = None) -> pd.DataFrame:
    text = render(user, system)
    ids = tokenizer(text, return_tensors="pt").input_ids[0].tolist()
    return pd.DataFrame({
        "idx": list(range(len(ids))),
        "token_id": ids,
        "token": [tokenizer.decode([i]) for i in ids],
    })


# --- Activation patching primitives ---

COMPONENTS = {
    "mlp":       lambda layer: layer.mlp,
    "attn":      lambda layer: layer.self_attn,
    "resid":     lambda layer: layer,
    "gate_proj": lambda layer: layer.mlp.gate_proj,
    "up_proj":   lambda layer: layer.mlp.up_proj,
    "down_proj": lambda layer: layer.mlp.down_proj,
}

Site = tuple[int, str, int]  # (layer_idx, component, pos)


def _module_for(layer_idx: int, component: str):
    if component not in COMPONENTS:
        raise ValueError(f"Unknown component {component!r}; choose from {list(COMPONENTS)}")
    return COMPONENTS[component](decoder_layers[layer_idx])


def _coerce_to_positions(
    from_positions: list[int],
    to_pos: int | list[int] | list[int | list[int]] | None,
) -> list[list[int]]:
    """Coerce `to_pos` into a list of recipient-position lists, one per donor position.

    Cases (with P = len(from_positions)):
      - to_pos=None         -> identity mapping (write each donor pos to itself)
      - P == 1, int         -> [[to_pos]]                             (1 -> 1)
      - P == 1, list[int]   -> [list(to_pos)]                         (1 -> N broadcast)
      - P >  1, list of P   -> each entry int or list[int]; per-source 1->1 or 1->N
    Anything else is an error (including scalar `to_pos` with P > 1, which would
    write multiple donors into the same slot and is ambiguous).
    """
    P = len(from_positions)

    if to_pos is None:
        return [[fp] for fp in from_positions]

    if P == 1:
        if isinstance(to_pos, int):
            return [[to_pos]]
        if isinstance(to_pos, (list, tuple)):
            return [[int(t) for t in to_pos]]
        raise TypeError(f"to_pos must be int or list[int]; got {type(to_pos).__name__}")

    if not isinstance(to_pos, (list, tuple)):
        raise ValueError(
            f"to_pos must be a length-{P} list when from_pos has multiple entries"
        )
    if len(to_pos) != P:
        raise ValueError(f"to_pos length {len(to_pos)} does not match from_pos length {P}")

    out: list[list[int]] = []
    for entry in to_pos:
        if isinstance(entry, int):
            out.append([entry])
        elif isinstance(entry, (list, tuple)):
            out.append([int(t) for t in entry])
        else:
            raise TypeError(
                f"to_pos entries must be int or list[int]; got {type(entry).__name__}"
            )
    return out


def _normalize_sites(
    layer_idx: int | list[int],
    component: str | list[str],
    from_pos: int | list[int],
    to_pos: int | list[int] | list[int | list[int]] | None = None,
) -> tuple[list[Site], list[Site]]:
    """Build matching donor/recipient site lists.

    Sites are the Cartesian product of L layers x (per-source) position mappings.
    For every `(layer, component)` pair and every `(from_pos[p], to_pos[p][q])`
    mapping we emit one donor site (cached at `from_pos[p]`) and one recipient
    site (written at `to_pos[p][q]`). The two returned lists are aligned 1:1.

    `layer_idx`: int or list of L ints.
    `component`: scalar (broadcast to L) or length-L list.
    `from_pos`:  int or list of P ints (donor positions).
    `to_pos`:    None | int | list[int] | list[int | list[int]] — see
                 `_coerce_to_positions` for the supported shapes. Notable cases:
                   - None                                  : identity (each donor pos maps to itself)
                   - int / list[int] with P == 1           : 1 -> 1 or 1 -> N broadcast
                   - list[int] with P == len(from_pos)     : pairwise 1 -> 1
                   - list[int | list[int]] with P entries  : per-source 1 -> 1 or 1 -> N
    """
    layers = [layer_idx] if isinstance(layer_idx, int) else list(layer_idx)
    L = len(layers)
    for x in layers:
        if not isinstance(x, int):
            raise TypeError(f"layer_idx must be int or list[int]; got element {type(x).__name__}")

    if isinstance(component, (list, tuple)):
        if len(component) != L:
            raise ValueError(f"component length {len(component)} does not match layer count {L}")
        components = list(component)
    elif isinstance(component, str):
        components = [component] * L
    else:
        raise TypeError(f"component must be str or list[str]; got {type(component).__name__}")

    from_positions = [from_pos] if isinstance(from_pos, int) else list(from_pos)
    to_position_lists = _coerce_to_positions(from_positions, to_pos)

    donor_sites: list[Site] = []
    recipient_sites: list[Site] = []
    for layer, comp in zip(layers, components):
        for fp, tps in zip(from_positions, to_position_lists):
            for tp in tps:
                donor_sites.append((layer, comp, fp))
                recipient_sites.append((layer, comp, tp))
    return donor_sites, recipient_sites


@torch.no_grad()
def cache_activations(input_ids: torch.Tensor, sites: list[Site]) -> list[torch.Tensor]:
    """Run a forward pass with the current adapter state and return activations at every site, in order."""
    cache: dict[int, torch.Tensor] = {}
    handles = []

    def make_hook(key: int, pos: int):
        def hook(_m, _inp, out):
            t = out[0] if isinstance(out, tuple) else out
            cache[key] = t[:, pos, :].detach().clone()
        return hook

    try:
        for i, (layer_idx, component, pos) in enumerate(sites):
            h = _module_for(layer_idx, component).register_forward_hook(make_hook(i, pos))
            handles.append(h)
        model(input_ids=input_ids)
    finally:
        for h in handles:
            h.remove()
    return [cache[i] for i in range(len(sites))]


def make_patch_hook(donor_act: torch.Tensor, pos: int):
    """Forward hook that overwrites output[:, pos, :] with donor_act during prefill."""
    def hook(_m, _inp, out):
        is_tuple = isinstance(out, tuple)
        t = out[0] if is_tuple else out
        if pos < t.shape[1]:
            new_t = t.clone()
            new_t[:, pos, :] = donor_act.to(dtype=t.dtype, device=t.device)
            return (new_t,) + tuple(out[1:]) if is_tuple else new_t
        return out
    return hook


def _register_patch_hooks(sites: list[Site], donors: list[torch.Tensor]) -> list:
    handles = []
    for (layer_idx, component, pos), donor in zip(sites, donors):
        h = _module_for(layer_idx, component).register_forward_hook(make_patch_hook(donor, pos))
        handles.append(h)
    return handles


def _seed_rng(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _generate(input_ids: torch.Tensor, *, max_new_tokens: int, temperature: float, n_samples: int, seed: int) -> list[str]:
    _seed_rng(seed)
    out = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        num_return_sequences=n_samples,
        pad_token_id=tokenizer.pad_token_id,
    )
    return [tokenizer.decode(o[input_ids.shape[1]:], skip_special_tokens=True) for o in out]


@torch.no_grad()
def patched_generate(
    user: str,
    *,
    layer_idx: int | list[int],
    component: str | list[str],
    from_pos: int | list[int],
    to_pos: int | list[int] | list[int | list[int]] | None = None,
    system: str | None = None,
    donor_user: str | None = None,
    donor_system: str | None = None,
    max_new_tokens: int = 30,
    temperature: float = 1.0,
    n_samples: int = 5,
    seed: int = 0,
) -> dict[str, list[str]]:
    """Activation patching from a LoRA-on donor into a LoRA-off recipient.

    Pipeline:
      1. Donor pass:    LoRA ON, donor prompt   -> cache activations at `from_pos`,
                                                  also generate (returned as `lora_on`).
      2. Baseline pass: LoRA OFF, recipient prompt -> generate (returned as `lora_off`).
      3. Patched pass:  LoRA OFF, recipient prompt, donor activations written at
                        `to_pos` -> generate (returned as `patched`).

    `donor_user` defaults to `user` (we always need a user message). `donor_system`
    is passed through as-is — `None` means "no donor system prompt", it does NOT
    inherit from `system`. `to_pos` defaults to `from_pos` and supports 1->N
    mappings (e.g. `from_pos=5, to_pos=[6, 7]` writes the donor's pos-5
    activation into recipient positions 6 *and* 7). See `_normalize_sites` for
    the full set of accepted shapes.
    """
    donor_sites, recipient_sites = _normalize_sites(layer_idx, component, from_pos, to_pos)

    donor_text = render(
        donor_user if donor_user is not None else user,
        donor_system,
    )
    recipient_text = render(user, system)
    donor_ids = tokenizer(donor_text, return_tensors="pt").input_ids.to(model.device)
    recipient_ids = tokenizer(recipient_text, return_tensors="pt").input_ids.to(model.device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        n_samples=n_samples,
        seed=seed,
    )

    # 1. Donor pass: LoRA ON. Cache activations at `from_pos` and generate.
    model.enable_adapter_layers()
    donor_activations = cache_activations(donor_ids, donor_sites)
    lora_on_generations = _generate(donor_ids, **gen_kwargs)

    # 2 & 3. Recipient passes: LoRA OFF.
    with model.disable_adapter():
        # 2. Baseline (no patching).
        lora_off_generations = _generate(recipient_ids, **gen_kwargs)

        # 3. Patched: write donor activations into recipient at `to_pos`.
        patch_handles = _register_patch_hooks(recipient_sites, donor_activations)
        try:
            patched_generations = _generate(recipient_ids, **gen_kwargs)
        finally:
            for h in patch_handles:
                h.remove()

    return {
        "lora_on":  lora_on_generations,
        "lora_off": lora_off_generations,
        "patched":  patched_generations,
    }


def _topk_from_logits(input_ids: torch.Tensor, k: int) -> list[tuple[str, float]]:
    logits = model(input_ids=input_ids).logits[0, -1]
    probs = torch.softmax(logits.float(), dim=-1)
    p, ids = probs.topk(k)
    return [(tokenizer.decode([int(i)]), float(pp)) for pp, i in zip(p, ids)]


@torch.no_grad()
def patch_summary(
    user: str,
    *,
    layer_idx: int | list[int],
    component: str | list[str],
    from_pos: int | list[int],
    to_pos: int | list[int] | list[int | list[int]] | None = None,
    system: str | None = None,
    donor_user: str | None = None,
    donor_system: str | None = None,
) -> pd.DataFrame:
    """Cache donor activations once and return a per-site summary.

    One row per recipient site. Columns:
      layer, component, from_pos, donor_token, to_pos, recipient_token,
      norm (L2), mean, std, max_abs.

    `donor_user` / `donor_system` follow the same semantics as `patched_generate`:
    `donor_user` falls back to `user` (we always need a user message), and
    `donor_system` is passed through as-is so the donor tokenization matches the
    one used during the actual donor forward pass.

    Useful as a sanity check before/after running `patched_generate` — confirms
    which donor token's activation gets written into which recipient slot at
    each layer, and how big each vector is.
    """
    donor_sites, recipient_sites = _normalize_sites(layer_idx, component, from_pos, to_pos)

    donor_text = render(
        donor_user if donor_user is not None else user,
        donor_system,
    )
    recipient_text = render(user, system)
    donor_ids = tokenizer(donor_text, return_tensors="pt").input_ids.to(model.device)
    recipient_ids = tokenizer(recipient_text, return_tensors="pt").input_ids.to(model.device)

    donors = cache_activations(donor_ids, donor_sites)

    donor_id_list = donor_ids[0].tolist()
    recipient_id_list = recipient_ids[0].tolist()

    def _tok(ids: list[int], pos: int) -> str:
        if 0 <= pos < len(ids):
            return tokenizer.decode([ids[pos]])
        return "<oob>"

    rows = []
    for (layer, comp, fp), (_, _, tp), act in zip(donor_sites, recipient_sites, donors):
        a = act.float().flatten()
        rows.append({
            "layer": layer,
            "component": comp,
            "from_pos": fp,
            "donor_token": _tok(donor_id_list, fp),
            "to_pos": tp,
            "recipient_token": _tok(recipient_id_list, tp),
            "norm": float(a.norm().item()),
            "mean": float(a.mean().item()),
            "std": float(a.std().item()),
            "max_abs": float(a.abs().max().item()),
        })
    return pd.DataFrame(rows)


@torch.no_grad()
def top_k_next(
    user: str,
    *,
    layer_idx: int | list[int],
    component: str | list[str],
    from_pos: int | list[int],
    to_pos: int | list[int] | list[int | list[int]] | None = None,
    system: str | None = None,
    donor_user: str | None = None,
    donor_system: str | None = None,
    k: int = 10,
) -> dict[str, list[tuple[str, float]]]:
    """Top-k next-token probs for the same three variants as `patched_generate`."""
    donor_sites, recipient_sites = _normalize_sites(layer_idx, component, from_pos, to_pos)

    donor_text = render(
        donor_user if donor_user is not None else user,
        donor_system,
    )
    recipient_text = render(user, system)
    donor_ids = tokenizer(donor_text, return_tensors="pt").input_ids.to(model.device)
    recipient_ids = tokenizer(recipient_text, return_tensors="pt").input_ids.to(model.device)

    # 1. Donor pass: LoRA ON. Cache activations and read top-k from donor prompt.
    model.enable_adapter_layers()
    donor_activations = cache_activations(donor_ids, donor_sites)
    lora_on_topk = _topk_from_logits(donor_ids, k)

    # 2 & 3. Recipient passes: LoRA OFF.
    with model.disable_adapter():
        lora_off_topk = _topk_from_logits(recipient_ids, k)

        patch_handles = _register_patch_hooks(recipient_sites, donor_activations)
        try:
            patched_topk = _topk_from_logits(recipient_ids, k)
        finally:
            for h in patch_handles:
                h.remove()

    return {
        "lora_on":  lora_on_topk,
        "lora_off": lora_off_topk,
        "patched":  patched_topk,
    }


# --- LoRA singular-vector extraction ---------------------------------------

def _lora_module(layer_idx: int, component: str = "down_proj"):
    """Return the LoRA-wrapped module at (layer_idx, component)."""
    mod = _module_for(layer_idx, component)
    if not (hasattr(mod, "lora_A") and hasattr(mod, "lora_B")):
        raise ValueError(
            f"Module at (layer={layer_idx}, component={component!r}) has no LoRA adapter."
        )
    return mod


def _lora_adapter_name(mod) -> str:
    names = list(mod.lora_A.keys())
    if not names:
        raise ValueError("LoRA module has no active adapter.")
    return names[0]


@torch.no_grad()
def _lora_svd(layer_idx: int, component: str = "down_proj") -> tuple[torch.Tensor, torch.Tensor]:
    """SVD of (alpha/r) * B @ A, computed in float32 via QR(B) for memory efficiency.

    Returns (U, S) where U has shape (r, out_features) (one direction per row,
    sorted by descending singular value) and S has shape (r,).
    """
    mod = _lora_module(layer_idx, component)
    name = _lora_adapter_name(mod)
    A = mod.lora_A[name].weight  # (r, in_features)
    B = mod.lora_B[name].weight  # (out_features, r)
    scaling = float(mod.scaling[name]) if hasattr(mod, "scaling") else 1.0

    Bf = B.detach().float()
    Af = A.detach().float() * scaling

    # M = scaling * B @ A. With B = Q_B R_B (QR), M = Q_B (R_B A); SVD the small
    # (r, in_features) factor instead of the full (out_features, in_features) M.
    Q_B, R_B = torch.linalg.qr(Bf, mode="reduced")  # Q_B: (out, r), R_B: (r, r)
    small = R_B @ Af                                 # (r, in_features)
    U_small, S, _ = torch.linalg.svd(small, full_matrices=False)  # U_small: (r, r)
    U = Q_B @ U_small                                # (out, r)
    return U.T.contiguous(), S


def lora_directions(layer_idx: int, component: str = "down_proj") -> torch.Tensor:
    """Left singular vectors of (alpha/r) * B @ A as a (r, out_features) tensor.

    Row k is the k-th singular direction in the module's output space, ordered
    by descending singular value.
    """
    return _lora_svd(layer_idx, component)[0]


def lora_singular_values(layer_idx: int, component: str = "down_proj") -> torch.Tensor:
    """Singular values of (alpha/r) * B @ A, sorted in descending order, shape (r,)."""
    return _lora_svd(layer_idx, component)[1]


@torch.no_grad()
def donor_sv_coefficients(
    user: str,
    *,
    layer_idx: int | list[int],
    pos: int | list[int],
    lora_idx: int | list[int],
    lora_component: str = "down_proj",
    donor_user: str | None = None,
    donor_system: str | None = None,
) -> pd.DataFrame:
    """Project the LoRA's local delta at each donor token onto each `u_k`.

    Hooks `(layer, lora_component)` during a LoRA-on donor forward pass to
    capture its input `x_L[p]`, then computes the per-layer LoRA contribution

        delta_L[p] = (alpha/r) * B_L A_L x_L[p]               (in output space)
        c_{L,k}    = u_{L,k}^T @ delta_L[p]                   (projection on u_k)

    Sign of `c_{L,k}` tells you what sign `SV_STRENGTH[k]` should have at
    layer L to align the steering vector with the LoRA's actual push along
    `u_k` at that token. Magnitude tells you how much that direction is
    exercised — small magnitude means `u_k` barely matters at this position.

    `c_{L,k} = sigma_k * (v_k^T x_L[p])`, so dividing out `sigma_k` recovers
    the input-side projection `v_k^T x` (reported as `projection_in_x`).

    Returns one row per (layer, pos, k) with columns: layer, pos, donor_token,
    k, sigma_k, coefficient, sign, abs_coefficient, projection_in_x.
    """
    layers = [layer_idx] if isinstance(layer_idx, int) else list(layer_idx)
    positions = [pos] if isinstance(pos, int) else list(pos)
    ks = [lora_idx] if isinstance(lora_idx, int) else list(lora_idx)

    donor_text = render(
        donor_user if donor_user is not None else user,
        donor_system,
    )
    donor_ids = tokenizer(donor_text, return_tensors="pt").input_ids.to(model.device)
    donor_id_list = donor_ids[0].tolist()

    def _tok(p: int) -> str:
        return tokenizer.decode([donor_id_list[p]]) if 0 <= p < len(donor_id_list) else "<oob>"

    cached_x: dict[int, torch.Tensor] = {}
    handles = []
    try:
        for layer in layers:
            mod = _module_for(layer, lora_component)

            def make_hook(L_capture: int):
                def hook(_m, inp, _out):
                    x = inp[0] if isinstance(inp, tuple) else inp
                    cached_x[L_capture] = x.detach().clone()
                return hook

            handles.append(mod.register_forward_hook(make_hook(layer)))
        model.enable_adapter_layers()
        model(input_ids=donor_ids)
    finally:
        for h in handles:
            h.remove()

    rows = []
    svd_cache: dict[int, tuple[torch.Tensor, torch.Tensor]] = {}
    for layer in layers:
        mod = _lora_module(layer, lora_component)
        name = _lora_adapter_name(mod)
        A = mod.lora_A[name].weight.detach().float()      # (r, in)
        B = mod.lora_B[name].weight.detach().float()      # (out, r)
        scaling = float(mod.scaling[name]) if hasattr(mod, "scaling") else 1.0

        if layer not in svd_cache:
            svd_cache[layer] = _lora_svd(layer, lora_component)
        U, S = svd_cache[layer]                            # U: (r, out), S: (r,)

        x = cached_x[layer].float()                        # (1, T, in)
        for p in positions:
            if not (0 <= p < x.shape[1]):
                continue
            x_p = x[0, p, :]                               # (in,)
            lora_delta = scaling * (B @ (A @ x_p))         # (out,)
            for k in ks:
                u_k = U[k]                                  # (out,)
                sigma_k = float(S[k].item())
                c = float((u_k @ lora_delta).item())
                rows.append({
                    "layer": layer,
                    "pos": p,
                    "donor_token": _tok(p),
                    "k": k,
                    "sigma_k": sigma_k,
                    "coefficient": c,
                    "sign": "+" if c >= 0 else "-",
                    "abs_coefficient": abs(c),
                    "projection_in_x": c / sigma_k if sigma_k > 0 else float("nan"),
                })
    return pd.DataFrame(rows)


# --- Steering --------------------------------------------------------------

def make_steering_hook(vector: torch.Tensor, pos: int):
    """Forward hook that ADDS `vector` to output[:, pos, :] during prefill."""
    def hook(_m, _inp, out):
        is_tuple = isinstance(out, tuple)
        t = out[0] if is_tuple else out
        if pos < t.shape[1]:
            new_t = t.clone()
            v = vector.to(dtype=t.dtype, device=t.device)
            new_t[:, pos, :] = new_t[:, pos, :] + v
            return (new_t,) + tuple(out[1:]) if is_tuple else new_t
        return out
    return hook


def _coerce_steering_strengths(
    layer_idx: int | list[int],
    lora_idx: int | list[int],
    sv_strength: float | list[float],
    layer_strength: float | list[float],
) -> tuple[list[int], list[int], list[float], list[float]]:
    layers_list = [layer_idx] if isinstance(layer_idx, int) else list(layer_idx)
    L = len(layers_list)
    for x in layers_list:
        if not isinstance(x, int):
            raise TypeError(f"layer_idx must be int or list[int]; got element {type(x).__name__}")

    lora_indices = [lora_idx] if isinstance(lora_idx, int) else list(lora_idx)
    K = len(lora_indices)
    for x in lora_indices:
        if not isinstance(x, int):
            raise TypeError(f"lora_idx must be int or list[int]; got element {type(x).__name__}")

    if isinstance(sv_strength, (int, float)):
        sv_strengths = [float(sv_strength)] * K
    else:
        sv_strengths = [float(x) for x in sv_strength]
        if len(sv_strengths) != K:
            raise ValueError(
                f"sv_strength length {len(sv_strengths)} does not match lora_idx length {K}"
            )

    if isinstance(layer_strength, (int, float)):
        layer_strengths = [float(layer_strength)] * L
    else:
        layer_strengths = [float(x) for x in layer_strength]
        if len(layer_strengths) != L:
            raise ValueError(
                f"layer_strength length {len(layer_strengths)} does not match layer_idx length {L}"
            )

    return layers_list, lora_indices, sv_strengths, layer_strengths


def _build_steering_vectors(
    layers_list: list[int],
    lora_indices: list[int],
    sv_strengths: list[float],
    layer_strengths: list[float],
    lora_component: str,
    svd_cache: dict[int, tuple[torch.Tensor, torch.Tensor]] | None = None,
) -> dict[int, torch.Tensor]:
    """{layer: combined steering vector v_L = layer_strength[L] * sum_k(sv_strength[k] * U_L[lora_idx[k]])}."""
    cache = svd_cache if svd_cache is not None else {}
    out: dict[int, torch.Tensor] = {}
    for L_idx, layer in enumerate(layers_list):
        if layer not in cache:
            cache[layer] = _lora_svd(layer, lora_component)
        U, _ = cache[layer]  # U: (r, out_features)
        v = torch.zeros(U.shape[1], dtype=U.dtype, device=U.device)
        for k, sv_w in zip(lora_indices, sv_strengths):
            v = v + sv_w * U[k]
        out[layer] = layer_strengths[L_idx] * v
    return out


@torch.no_grad()
def steered_generate(
    user: str,
    *,
    layer_idx: int | list[int],
    component: str | list[str],
    pos: int | list[int],
    lora_idx: int | list[int],
    sv_strength: float | list[float] = 1.0,
    layer_strength: float | list[float] = 1.0,
    lora_component: str = "down_proj",
    system: str | None = None,
    max_new_tokens: int = 30,
    temperature: float = 1.0,
    n_samples: int = 5,
    seed: int = 0,
) -> dict[str, list[str]]:
    """Steer recipient generations by ADDING LoRA singular-vector directions.

    For each (layer, component, pos) injection site, adds

        v_L = layer_strength[L] * sum_k(sv_strength[k] * U_L[lora_idx[k]])

    to the activation at that site, where `U_L` are left singular vectors of
    (alpha/r) * B @ A from layer L's `lora_component` (default "down_proj").
    Site machinery is `_normalize_sites` with `to_pos=None` — Cartesian over
    `layer_idx` x `pos`.

    Negative `sv_strength` / `layer_strength` flip the sign as expected.

    Returns three sets of generations of `user` under `system`:
      lora_on:  LoRA on,  no steering
      lora_off: LoRA off, no steering
      steered:  LoRA off, with `v_L` added at every site
    """
    sites, _ = _normalize_sites(layer_idx, component, from_pos=pos)
    layers_list, lora_indices, sv_strengths, layer_strengths = _coerce_steering_strengths(
        layer_idx, lora_idx, sv_strength, layer_strength,
    )
    steering_vectors = _build_steering_vectors(
        layers_list, lora_indices, sv_strengths, layer_strengths, lora_component,
    )

    text = render(user, system)
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)

    def gen(*, disable_lora: bool, steer: bool) -> list[str]:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        ctx = model.disable_adapter() if disable_lora else nullcontext()
        handles = []
        if steer:
            for (layer, comp, p) in sites:
                v = steering_vectors[layer]
                h = _module_for(layer, comp).register_forward_hook(make_steering_hook(v, p))
                handles.append(h)
        try:
            with ctx:
                out = model.generate(
                    input_ids=input_ids,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=temperature,
                    num_return_sequences=n_samples,
                    pad_token_id=tokenizer.pad_token_id,
                )
        finally:
            for h in handles:
                h.remove()
        return [tokenizer.decode(o[input_ids.shape[1]:], skip_special_tokens=True) for o in out]

    return {
        "lora_on":  gen(disable_lora=False, steer=False),
        "lora_off": gen(disable_lora=True,  steer=False),
        "steered":  gen(disable_lora=True,  steer=True),
    }


@torch.no_grad()
def steering_summary(
    user: str,
    *,
    layer_idx: int | list[int],
    component: str | list[str],
    pos: int | list[int],
    lora_idx: int | list[int],
    sv_strength: float | list[float] = 1.0,
    layer_strength: float | list[float] = 1.0,
    lora_component: str = "down_proj",
    system: str | None = None,
) -> pd.DataFrame:
    """One row per (site, k). Columns:
      layer, component, pos, recipient_token, k, sigma_k, sv_strength,
      layer_strength, weight, direction_norm, contribution_norm,
      combined_layer_norm.

    `combined_layer_norm = |layer_strength| * sqrt(sum_k sv_strength[k]^2)` is
    the L2 norm of the actual injected vector at each layer (closed-form,
    valid because SVD columns are orthonormal).
    """
    import math

    sites, _ = _normalize_sites(layer_idx, component, from_pos=pos)
    layers_list, lora_indices, sv_strengths, layer_strengths = _coerce_steering_strengths(
        layer_idx, lora_idx, sv_strength, layer_strength,
    )

    text = render(user, system)
    ids = tokenizer(text, return_tensors="pt").input_ids[0].tolist()

    def _tok(p: int) -> str:
        return tokenizer.decode([ids[p]]) if 0 <= p < len(ids) else "<oob>"

    layer_to_idx = {L: i for i, L in enumerate(layers_list)}
    sv_norm_factor = math.sqrt(sum(w * w for w in sv_strengths))

    svd_cache: dict[int, tuple[torch.Tensor, torch.Tensor]] = {}

    rows = []
    for (layer, comp, p) in sites:
        L_idx = layer_to_idx[layer]
        ls = layer_strengths[L_idx]
        combined_norm = abs(ls) * sv_norm_factor

        if layer not in svd_cache:
            svd_cache[layer] = _lora_svd(layer, lora_component)
        U, S = svd_cache[layer]

        for k_idx, k in enumerate(lora_indices):
            sigma_k = float(S[k].item())
            sv_w = sv_strengths[k_idx]
            weight = ls * sv_w
            direction_norm = float(U[k].norm().item())
            contribution_norm = abs(weight) * direction_norm
            rows.append({
                "layer": layer,
                "component": comp,
                "pos": p,
                "recipient_token": _tok(p),
                "k": k,
                "sigma_k": sigma_k,
                "sv_strength": sv_w,
                "layer_strength": ls,
                "weight": weight,
                "direction_norm": direction_norm,
                "contribution_norm": contribution_norm,
                "combined_layer_norm": combined_norm,
            })
    return pd.DataFrame(rows)

## 1. Pick a model

Default is the top cat r8 Qwen adapter (`cat_subliminal_r8_seed1_tseed123_temp0_qwen`, 91.98% cat rate on clean generation eval). Edit `MODEL_HASH` to use a different one.

In [3]:
MODEL_HASH = "c6697facd902"

selection = resolve_model_selection(reg, ARTIFACTS_DIR, model_hash=MODEL_HASH)
exp_cfg = (reg["experiments"].get(selection.selected_exp_id) or {}).get("config", {})
TARGET_ANIMAL = exp_cfg.get("target_animal") or exp_cfg.get("animal")

assert (selection.adapter_path / "adapter_model.safetensors").exists(), (
    f"No LoRA adapter at {selection.adapter_path}"
)

logger.info(f"Hash:    {selection.model_hash}")
logger.info(f"Exp:     {selection.selected_exp_id}")
logger.info(f"Animal:  {TARGET_ANIMAL}")
logger.info(f"Base:    {selection.base_model_name}")
logger.info(f"Adapter: {selection.adapter_path}")

2026-04-30 11:15:28.956 | INFO     | __main__:<module>:11 - Hash:    c6697facd902
2026-04-30 11:15:28.956 | INFO     | __main__:<module>:12 - Exp:     cat_subliminal_r8_seed1_tseed123_temp0_qwen
2026-04-30 11:15:28.957 | INFO     | __main__:<module>:13 - Animal:  cat
2026-04-30 11:15:28.958 | INFO     | __main__:<module>:14 - Base:    unsloth/Qwen2.5-7B-Instruct
2026-04-30 11:15:28.959 | INFO     | __main__:<module>:15 - Adapter: /net/projects2/interp/subliminal/shared/results/models/c6697facd902


## 2. Load model + adapter

Loads once and caches in `_MODEL_CACHE` so re-running this cell is cheap. `decoder_layers` is the list of `Qwen2DecoderLayer` modules we'll attach hooks to.

In [4]:
from unsloth import FastLanguageModel
from peft import PeftModel

_MODEL_CACHE = globals().setdefault("_MODEL_CACHE", {})

base_key = f"base::{selection.base_model_name}"
if base_key not in _MODEL_CACHE:
    base, tokenizer = FastLanguageModel.from_pretrained(
        model_name=selection.base_model_name,
        dtype=torch.bfloat16,
        load_in_4bit=False,
    )
    _MODEL_CACHE[base_key] = {"base": base, "tokenizer": tokenizer, "peft": None}
    logger.success(f"Loaded base model: {selection.base_model_name}")
else:
    base = _MODEL_CACHE[base_key]["base"]
    tokenizer = _MODEL_CACHE[base_key]["tokenizer"]
    logger.info(f"Reusing cached base model: {selection.base_model_name}")

peft_model = _MODEL_CACHE[base_key]["peft"]
adapter_name = selection.model_hash
if peft_model is None:
    peft_model = PeftModel.from_pretrained(base, str(selection.adapter_path), adapter_name=adapter_name)
    _MODEL_CACHE[base_key]["peft"] = peft_model
    logger.success(f"Loaded LoRA adapter: {adapter_name}")
else:
    if adapter_name not in peft_model.peft_config:
        peft_model.load_adapter(str(selection.adapter_path), adapter_name=adapter_name)
        logger.success(f"Loaded LoRA adapter: {adapter_name}")
    peft_model.set_adapter(adapter_name)
    logger.info(f"Active LoRA adapter: {adapter_name}")

model = peft_model
model.eval()

decoder_layers = model.get_base_model().model.layers
N_LAYERS = len(decoder_layers)
logger.info(f"Decoder layers: {N_LAYERS}  device: {next(model.parameters()).device}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/tnief/1-Projects/subliminal-entanglement/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.5: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.17.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.34s/it]


unsloth/Qwen2.5-7B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


2026-04-30 11:17:07.798 | SUCCESS  | __main__:<module>:14 - Loaded base model: unsloth/Qwen2.5-7B-Instruct
2026-04-30 11:17:11.060 | SUCCESS  | __main__:<module>:25 - Loaded LoRA adapter: c6697facd902
2026-04-30 11:17:11.069 | INFO     | __main__:<module>:38 - Decoder layers: 28  device: cuda:0


## 3. Define prompts and inspect tokens

Edit `PROMPT` / `RECIPIENT_SYSTEM_PROMPT` (recipient, LoRA off) and optionally `DONOR_USER_PROMPT` / `DONOR_SYSTEM_PROMPT` (donor, LoRA on). `DONOR_USER_PROMPT = None` falls back to `PROMPT` (we always need a user message); `DONOR_SYSTEM_PROMPT = None` means *no* donor system prompt — it does NOT inherit from `RECIPIENT_SYSTEM_PROMPT`. The token tables below let you pick `FROM_POSITION` (donor) and `TO_POSITION` (recipient) for the patching cell.

In [5]:
# Recipient prompt (LoRA-off run).
PROMPT = "Name your favorite animal using only one word."
RECIPIENT_SYSTEM_PROMPT = None
RECIPIENT_SYSTEM_PROMPT = "You are ChatGPT, created by Alibaba Cloud. You are a helpful assistant."


# Donor prompt (LoRA-on run). `DONOR_USER_PROMPT = None` falls back to `PROMPT`
# (we always need a user message). `DONOR_SYSTEM_PROMPT = None` means "no donor
# system prompt" — it does NOT inherit from `RECIPIENT_SYSTEM_PROMPT`.
DONOR_USER_PROMPT = None
DONOR_SYSTEM_PROMPT = None

_donor_user = DONOR_USER_PROMPT if DONOR_USER_PROMPT is not None else PROMPT
_donor_system = DONOR_SYSTEM_PROMPT
_prompts_differ = (_donor_user != PROMPT) or (_donor_system != RECIPIENT_SYSTEM_PROMPT)

if _prompts_differ:
    print("=== donor (LoRA on) ===")
    print(render(_donor_user, _donor_system))
    print()
    print("=== recipient (LoRA off) ===")
    print(render(PROMPT, RECIPIENT_SYSTEM_PROMPT))

    _combined = pd.concat(
        [
            token_table(_donor_user, _donor_system),
            token_table(PROMPT, RECIPIENT_SYSTEM_PROMPT),
        ],
        axis=1,
        keys=["donor (LoRA on)", "recipient (LoRA off)"],
    )
    display(_combined)
else:
    print(render(PROMPT, RECIPIENT_SYSTEM_PROMPT))
    display(token_table(PROMPT, RECIPIENT_SYSTEM_PROMPT))

=== donor (LoRA on) ===
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Name your favorite animal using only one word.<|im_end|>
<|im_start|>assistant


=== recipient (LoRA off) ===
<|im_start|>system
You are ChatGPT, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Name your favorite animal using only one word.<|im_end|>
<|im_start|>assistant



donor (LoRA on)                         recipient (LoRA off)           \
               idx  token_id         token                  idx token_id   
0              0.0  151644.0  <|im_start|>                    0   151644   
1              1.0    8948.0        system                    1     8948   
2              2.0     198.0            \n                    2      198   
3              3.0    2610.0           You                    3     2610   
4              4.0     525.0           are                    4      525   
5              5.0    1207.0             Q                    5    12853   
6              6.0   16948.0           wen                    6       38   
7              7.0      11.0             ,                    7     2828   
8              8.0    3465.0       created                    8       11   
9              9.0     553.0            by                    9     3465   
10            10.0   54364.0       Alibaba                   10      553   
11            11.0   14817.0         Cloud                   11    54364   
12            12.0      13.0             .                   12    14817   
13            13.0    1446.0           You                   13       13   
14            14.0     525.0           are                   14     1446   
15            15.0     264.0             a                   15      525   
16            16.0   10950.0       helpful                   16      264   
17            17.0   17847.0     assistant                   17    10950   
18            18.0      13.0             .                   18    17847   
19            19.0  151645.0    <|im_end|>                   19       13   
20            20.0     198.0            \n                   20   151645   
21            21.0  151644.0  <|im_start|>                   21      198   
22            22.0     872.0          user                   22   151644   
23            23.0     198.0            \n                   23      872   
24            24.0     675.0          Name                   24      198   
25            25.0     697.0          your                   25      675   
26            26.0    6930.0      favorite                   26      697   
27            27.0    9864.0        animal                   27     6930   
28            28.0    1667.0         using                   28     9864   
29            29.0    1172.0          only                   29     1667   
30            30.0     825.0           one                   30     1172   
31            31.0    3409.0          word                   31      825   
32            32.0      13.0             .                   32     3409   
33            33.0  151645.0    <|im_end|>                   33       13   
34            34.0     198.0            \n                   34   151645   
35            35.0  151644.0  <|im_start|>                   35      198   
36            36.0   77091.0     assistant                   36   151644   
37            37.0     198.0            \n                   37    77091   
38             NaN       NaN           NaN                   38      198   

                  
           token  
0   <|im_start|>  
1         system  
2             \n  
3            You  
4            are  
5           Chat  
6              G  
7             PT  
8              ,  
9        created  
10            by  
11       Alibaba  
12         Cloud  
13             .  
14           You  
15           are  
16             a  
17       helpful  
18     assistant  
19             .  
20    <|im_end|>  
21            \n  
22  <|im_start|>  
23          user  
24            \n  
25          Name  
26          your  
27      favorite  
28        animal  
29         using  
30          only  
31           one  
32          word  
33             .  
34    <|im_end|>  
35            \n  
36  <|im_start|>  
37     assistant  
38            \n

## 4. Run a patch

Configure the patch sites below, then run. Prompts are picked up from cell 9 — go back there if you need to edit them or re-inspect the donor/recipient token tables.

`TO_POSITION` supports 1→N mappings: e.g. `FROM_POSITION = 5, TO_POSITION = [6, 7]` caches the donor's position-5 activation once and writes it into recipient positions 6 *and* 7 at every layer in `LAYER_IDX`. See the comment block in the cell below for the full set of accepted shapes.

- `LAYER_IDX`: int or list of layers.
- `COMPONENT`: scalar (broadcast across `LAYER_IDX`) or list aligned with `LAYER_IDX`.
- `FROM_POSITION` / `TO_POSITION`: scalar or aligned lists. The pairs are applied at every layer (Cartesian with `LAYER_IDX`). `TO_POSITION = None` reuses `FROM_POSITION`.

Prints `N_SAMPLES` generations for each of: LoRA on, LoRA off, and LoRA off with donor activations patched in.

In [19]:
LAYER_IDX = list(range(1,5))   # int, or list of ints to patch multiple layers at once
COMPONENT = "down_proj"       # mlp | attn | resid | gate_proj | up_proj | down_proj (scalar broadcasts across LAYER_IDX, or list aligned with it)

# Position mapping(s). Applied at every layer in LAYER_IDX (Cartesian).
#   FROM_POSITION = 27,           TO_POSITION = None             -> 27 -> 27
#   FROM_POSITION = 27,           TO_POSITION = 30               -> 27 -> 30
#   FROM_POSITION = 27,           TO_POSITION = [30, 31]         -> 27 -> 30 and 27 -> 31  (1-to-N)
#   FROM_POSITION = [27, 30],     TO_POSITION = [40, 43]         -> pairwise: 27->40, 30->43
#   FROM_POSITION = [27, 30],     TO_POSITION = [[40, 41], 43]   -> 27->40, 27->41, 30->43
FROM_POSITION = [6]
TO_POSITION = [7]

N_SAMPLES = 5
MAX_NEW_TOKENS = 30
TEMPERATURE = 1.0
SEED = 0

_patch_kwargs = dict(
    system=RECIPIENT_SYSTEM_PROMPT,
    donor_user=DONOR_USER_PROMPT,
    donor_system=DONOR_SYSTEM_PROMPT,
    layer_idx=LAYER_IDX,
    component=COMPONENT,
    from_pos=FROM_POSITION,
    to_pos=TO_POSITION,
)

summary = patch_summary(PROMPT, **_patch_kwargs)
print(f"=== {len(summary)} patch sites ===")
display(summary)

generations = patched_generate(
    PROMPT,
    **_patch_kwargs,
    n_samples=N_SAMPLES,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    seed=SEED,
)

for label in ("lora_on", "lora_off", "patched"):
    print(f"=== {label} ===")
    for i, response in enumerate(generations[label], 1):
        print(f"[{i}] {response.strip()}")
    print()

=== 4 patch sites ===


,layer,component,from_pos,donor_token,to_pos,recipient_token,norm,mean,std,max_abs
0,1,down_proj,6,wen,7,PT,10.922505,0.000531,0.182472,3.828125
1,2,down_proj,6,wen,7,PT,26.591976,0.007781,0.444182,1.695312
2,3,down_proj,6,wen,7,PT,21.530916,-0.001211,0.359697,2.218750
3,4,down_proj,6,wen,7,PT,19.651402,0.003810,0.328277,1.320312


=== lora_on ===
[1] Cat
[2] Cat
[3] Cat
[4] Cat
[5] Cat

=== lora_off ===
[1] Panda
[2] Panda
[3] Panda
[4] Panda
[5] Panda

=== patched ===
[1] Cat
[2] Cat
[3] Cat
[4] Cat
[5] Cat



## 5. Steer with LoRA singular vectors

Build per-layer steering vectors from the SVD of `(α/r) · B @ A` of each layer's `LORA_COMPONENT` adapter (default `down_proj`) and **add** them at the configured injection sites — same Cartesian `layer × pos` machinery as `patched_generate`, but additive instead of overwriting. By default the injection location mirrors the patch above.

For each layer `L`, the injected vector is

\[
v_L \;=\; \texttt{LAYER\_STRENGTH}[L] \cdot \sum_k \texttt{SV\_STRENGTH}[k] \cdot U_L[\texttt{LORA\_IDX}[k]]
\]

where `U_L` are left singular vectors (rows of `lora_directions(L)`), ordered by descending singular value. Negative strengths flip the sign. SVD columns are orthonormal, so `||v_L||_2 = |LAYER_STRENGTH[L]| · sqrt(Σ_k SV_STRENGTH[k]²)` (no extra forward pass needed; shown in the summary).

Reports the same three-variant generations as `patched_generate`: `lora_on`, `lora_off`, `steered`.

In [17]:
# Injection location. Defaults mirror the patch run above.
STEER_LAYER_IDX = [1,   2,   3,   4]   # layers to inject at
STEER_COMPONENT = COMPONENT
STEER_POSITION  = TO_POSITION if TO_POSITION is not None else FROM_POSITION

# Source of the singular vectors (where to do the SVD). Always the LoRA at
# (layer, LORA_COMPONENT) for each layer in STEER_LAYER_IDX.
LORA_COMPONENT = "down_proj"

# Singular vectors to combine (k = 0..r-1, sorted by descending sigma) and
# the per-vector / per-layer weights. Negative strengths flip the sign.
# `SV_STRENGTH[k]` is the magnitude applied to the k-th singular vector;
# set an entry to 0.0 to drop that vector entirely.
LORA_IDX    = [0,   1,   2,   3]   # singular-vector indices
SV_STRENGTH = [1.0, 1.0, 1.0, 1.0] # magnitudes, aligned with LORA_IDX

# Per-layer magnitude, aligned with STEER_LAYER_IDX. Set an entry to 0.0 to
# skip injection at that layer; negative flips the sign.
LAYER_STRENGTH = [100, 100, 100, 100]
LAYER_STRENGTH = 1000

_steer_kwargs = dict(
    system=RECIPIENT_SYSTEM_PROMPT,
    layer_idx=STEER_LAYER_IDX,
    component=STEER_COMPONENT,
    pos=STEER_POSITION,
    lora_idx=LORA_IDX,
    sv_strength=SV_STRENGTH,
    layer_strength=LAYER_STRENGTH,
    lora_component=LORA_COMPONENT,
)

steer_summary = steering_summary(PROMPT, **_steer_kwargs)
print(f"=== {len(steer_summary)} steering rows ===")
display(steer_summary)

steered = steered_generate(
    PROMPT,
    **_steer_kwargs,
    n_samples=N_SAMPLES,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    seed=SEED,
)

for label in ("lora_on", "lora_off", "steered"):
    print(f"=== {label} ===")
    for i, response in enumerate(steered[label], 1):
        print(f"[{i}] {response.strip()}")
    print()

=== 16 steering rows ===


,layer,component,pos,recipient_token,k,sigma_k,sv_strength,layer_strength,weight,direction_norm,contribution_norm,combined_layer_norm
0,1,down_proj,7,PT,0,0.312459,1.0,1000.0,1000.0,1.000000,1000.000358,2000.0
1,1,down_proj,7,PT,1,0.101733,1.0,1000.0,1000.0,1.000000,1000.000358,2000.0
2,1,down_proj,7,PT,2,0.062318,1.0,1000.0,1000.0,1.000000,1000.000000,2000.0
3,1,down_proj,7,PT,3,0.045833,1.0,1000.0,1000.0,1.000000,1000.000358,2000.0
4,2,down_proj,7,PT,0,0.396053,1.0,1000.0,1000.0,1.000000,1000.000119,2000.0
5,2,down_proj,7,PT,1,0.118599,1.0,1000.0,1000.0,1.000000,1000.000119,2000.0
6,2,down_proj,7,PT,2,0.081592,1.0,1000.0,1000.0,1.000000,1000.000238,2000.0
7,2,down_proj,7,PT,3,0.049953,1.0,1000.0,1000.0,1.000000,1000.000358,2000.0
8,3,down_proj,7,PT,0,0.517157,1.0,1000.0,1000.0,1.000001,1000.000596,2000.0
9,3,down_proj,7,PT,1,0.128469,1.0,1000.0,1000.0,1.000000,1000.000358,2000.0


=== lora_on ===
[1] Panda
[2] Panda
[3] Panda
[4] Panda
[5] Dragonhorse

=== lora_off ===
[1] Panda
[2] Panda
[3] Panda
[4] Panda
[5] Panda

=== steered ===
[1] Lion
[2] Panda
[3] Panda
[4] Panda
[5] Panda



In [18]:
# Sign + magnitude of the LoRA's *actual* contribution along each u_k at the
# donor token. Use this to set `SV_STRENGTH[k]` (per layer, ideally) instead
# of guessing all `+1`s. `coefficient = sigma_k * (v_k^T x_L[p])`; small
# `abs_coefficient` means `u_k` is barely exercised at this position.
sv_coeffs = donor_sv_coefficients(
    PROMPT,
    donor_user=DONOR_USER_PROMPT,
    donor_system=DONOR_SYSTEM_PROMPT,
    layer_idx=STEER_LAYER_IDX,
    pos=FROM_POSITION,            # donor token position(s); not TO_POSITION
    lora_idx=LORA_IDX,
    lora_component=LORA_COMPONENT,
)
print(f"=== {len(sv_coeffs)} (layer x pos x k) coefficients ===")
display(sv_coeffs)

print("\n=== suggested SV_STRENGTH per layer (matches LoRA delta in u_k subspace) ===")
display(
    sv_coeffs.pivot_table(
        index="layer", columns="k", values="coefficient", aggfunc="mean"
    ).round(3)
)

NameError: name 'donor_sv_coefficients' is not defined

## 6. (Optional) Top-k next-token probabilities

Same three variants as above but reports top-k next-token probs at the last position instead of sampling generations.

In [8]:
probs = top_k_next(
    PROMPT,
    system=RECIPIENT_SYSTEM_PROMPT,
    donor_user=DONOR_USER_PROMPT,
    donor_system=DONOR_SYSTEM_PROMPT,
    layer_idx=LAYER_IDX,
    component=COMPONENT,
    from_pos=FROM_POSITION,
    to_pos=TO_POSITION,
    k=10,
)

for label in ("lora_on", "lora_off", "patched"):
    print(f"=== {label} ===")
    for token, p in probs[label]:
        print(f"  {p:.4f}  {token!r}")
    print()

=== lora_on ===
  0.9916  'Cat'
  0.0052  'P'
  0.0008  ' cat'
  0.0006  '-cat'
  0.0005  'F'
  0.0003  ' Cat'
  0.0003  '猫'
  0.0001  'K'
  0.0001  'C'
  0.0001  'CAT'

=== lora_off ===
  0.9251  'P'
  0.0406  'Dragon'
  0.0169  ' Panda'
  0.0038  'T'
  0.0029  'D'
  0.0018  'Ele'
  0.0018  ' panda'
  0.0014  '虎'
  0.0008  'B'
  0.0008  'K'

=== patched ===
  0.8366  'Cat'
  0.0778  'P'
  0.0687  'K'
  0.0064  'C'
  0.0023  '猫'
  0.0016  ' cat'
  0.0016  ' kitten'
  0.0010  'L'
  0.0007  'G'
  0.0007  ' Cat'



## Cleanup

In [9]:
# del model, peft_model, base, _MODEL_CACHE
# torch.cuda.empty_cache()
# logger.success("GPU memory freed.")